# Visualisation & Styling with OMOPy

This notebook covers `omopy.vis` — the Python equivalent of the R `visOmopResults` package.

We demonstrate:
- **Format functions** — numeric formatting, estimate names, headers, min cell count suppression
- **Table styling** — `TableStyle`, `vis_omop_table`, `vis_table`, `format_table`
- **Plot styling** — `PlotStyle`, `scatter_plot`, `bar_plot`, `box_plot`
- **Text customisation** — `customise_text` for label formatting
- **Mock data** — `mock_summarised_result` for testing without a database

No database connection is needed — everything runs on mock `SummarisedResult` objects.

## 1. Setup

In [1]:
import polars as pl

from omopy.vis import (
    # Format pipeline
    format_estimate_value,
    format_estimate_name,
    format_header,
    format_min_cell_count,
    format_table,
    # Tidy helpers
    tidy_columns,
    tidy_result,
    # High-level tables
    vis_omop_table,
    vis_table,
    # Plots
    scatter_plot,
    bar_plot,
    box_plot,
    # Style
    TableStyle,
    PlotStyle,
    customise_text,
    default_table_style,
    default_plot_style,
    # Mock data
    mock_summarised_result,
)

# Also import mock_cohort_characteristics to show vis with real-ish results
from omopy.characteristics import mock_cohort_characteristics

print("All 19 omopy.vis exports loaded successfully.")

All 19 omopy.vis exports loaded successfully.


## 2. Creating Mock Data

`mock_summarised_result(n_cohorts, n_strata)` generates a `SummarisedResult` with
cohort groups, strata (overall, sex, age_group), and variables (counts, age, medications).

In [2]:
# Default: 2 cohorts, 3 strata
result = mock_summarised_result()
print(f"Rows: {result.data.shape[0]}, Columns: {result.data.shape[1]}")
print(f"Settings columns: {result.settings.columns}")
print(result.data.head(5))

Rows: 30, Columns: 13
Settings columns: ['result_id', 'result_type', 'package_name', 'package_version', 'min_cell_count']
shape: (5, 13)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ result_id ┆ cdm_name ┆ group_nam ┆ group_lev ┆ … ┆ estimate_ ┆ estimate_ ┆ additiona ┆ additiona │
│ ---       ┆ ---      ┆ e         ┆ el        ┆   ┆ type      ┆ value     ┆ l_name    ┆ l_level   │
│ i64       ┆ str      ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆          ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1         ┆ mock_cdm ┆ cohort_na ┆ cohort_1  ┆ … ┆ integer   ┆ 377       ┆ overall   ┆ overall   │
│           ┆          ┆ me        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 1         ┆ mock_cdm ┆ cohort_na ┆ cohort_1  ┆ … ┆ nu

In [3]:
# More cohorts, single stratum
result_small = mock_summarised_result(n_cohorts=3, n_strata=1)
print(f"3 cohorts, 1 stratum -> {result_small.data.shape[0]} rows")
print(result_small.data.select("group_level", "strata_name", "variable_name", "estimate_name", "estimate_value").head(10))

3 cohorts, 1 stratum -> 15 rows
shape: (10, 5)
┌─────────────┬─────────────┬─────────────────┬───────────────┬────────────────┐
│ group_level ┆ strata_name ┆ variable_name   ┆ estimate_name ┆ estimate_value │
│ ---         ┆ ---         ┆ ---             ┆ ---           ┆ ---            │
│ str         ┆ str         ┆ str             ┆ str           ┆ str            │
╞═════════════╪═════════════╪═════════════════╪═══════════════╪════════════════╡
│ cohort_1    ┆ overall     ┆ number subjects ┆ count         ┆ 377            │
│ cohort_1    ┆ overall     ┆ age             ┆ mean          ┆ 34.45          │
│ cohort_1    ┆ overall     ┆ age             ┆ sd            ┆ 16.12          │
│ cohort_1    ┆ overall     ┆ Medications     ┆ count         ┆ 135            │
│ cohort_1    ┆ overall     ┆ Medications     ┆ percentage    ┆ 35.81          │
│ cohort_2    ┆ overall     ┆ number subjects ┆ count         ┆ 164            │
│ cohort_2    ┆ overall     ┆ age             ┆ mean          

In [4]:
# Using mock_cohort_characteristics for richer data (demographics + age quantiles)
chars = mock_cohort_characteristics(n_cohorts=2, n_strata=1)
print(f"Characteristics mock: {chars.data.shape[0]} rows")
print(chars.data.select("variable_name", "estimate_name", "estimate_value").head(10))

Characteristics mock: 120 rows
shape: (10, 3)
┌─────────────────┬───────────────┬────────────────┐
│ variable_name   ┆ estimate_name ┆ estimate_value │
│ ---             ┆ ---           ┆ ---            │
│ str             ┆ str           ┆ str            │
╞═════════════════╪═══════════════╪════════════════╡
│ Number records  ┆ count         ┆ 405            │
│ Number subjects ┆ count         ┆ 377            │
│ Age             ┆ mean          ┆ 36.00          │
│ Age             ┆ sd            ┆ 11.30          │
│ Age             ┆ min           ┆ 2.10           │
│ Age             ┆ q25           ┆ 24.70          │
│ Age             ┆ median        ┆ 36.00          │
│ Age             ┆ q75           ┆ 47.30          │
│ Age             ┆ max           ┆ 69.90          │
│ Sex             ┆ count         ┆ 170            │
└─────────────────┴───────────────┴────────────────┘


## 3. Formatting Values

The format pipeline transforms `SummarisedResult` objects in place (returning new copies).

### format_estimate_value

Formats `estimate_value` based on `estimate_type` — applies decimal rounding and big marks.

```python
format_estimate_value(result, *, decimals=None, decimal_mark=".", big_mark=",")
```

In [5]:
formatted = format_estimate_value(result)

# Compare before and after for a numeric estimate
before = result.data.filter(
    (pl.col("estimate_name") == "mean") & (pl.col("variable_name") == "age")
).select("estimate_value").head(3)

after = formatted.data.filter(
    (pl.col("estimate_name") == "mean") & (pl.col("variable_name") == "age")
).select("estimate_value").head(3)

print("Before formatting:")
print(before)
print("\nAfter formatting (2 decimal places for numeric):")
print(after)

Before formatting:
shape: (3, 1)
┌────────────────┐
│ estimate_value │
│ ---            │
│ str            │
╞════════════════╡
│ 34.45          │
│ 35.58          │
│ 53.62          │
└────────────────┘

After formatting (2 decimal places for numeric):
shape: (3, 1)
┌────────────────┐
│ estimate_value │
│ ---            │
│ str            │
╞════════════════╡
│ 34.45          │
│ 35.58          │
│ 53.62          │
└────────────────┘


In [6]:
# Custom decimals per estimate type
custom_fmt = format_estimate_value(result, decimals={"numeric": 1, "percentage": 0})

print("Custom decimals — numeric=1, percentage=0:")
print(
    custom_fmt.data.filter(pl.col("estimate_type").is_in(["numeric", "percentage"]))
    .select("estimate_type", "estimate_name", "estimate_value")
    .head(6)
)

Custom decimals — numeric=1, percentage=0:
shape: (6, 3)
┌───────────────┬───────────────┬────────────────┐
│ estimate_type ┆ estimate_name ┆ estimate_value │
│ ---           ┆ ---           ┆ ---            │
│ str           ┆ str           ┆ str            │
╞═══════════════╪═══════════════╪════════════════╡
│ numeric       ┆ mean          ┆ 34.5           │
│ numeric       ┆ sd            ┆ 16.1           │
│ percentage    ┆ percentage    ┆ 36             │
│ numeric       ┆ mean          ┆ 35.6           │
│ numeric       ┆ sd            ┆ 6.5            │
│ percentage    ┆ percentage    ┆ 91             │
└───────────────┴───────────────┴────────────────┘


### format_estimate_name

Combines estimates using template patterns. E.g. merge `count` and `percentage` into `"N (%)"`.

```python
format_estimate_name(result, *, estimate_name=None, keep_not_formatted=True, use_format_order=True)
```

In [7]:
combined = format_estimate_value(result)
combined = format_estimate_name(
    combined,
    estimate_name={"N (%)": "<count> (<percentage>%)", "Mean (SD)": "<mean> (<sd>)"},
)

print("Combined estimate names:")
print(
    combined.data.select("variable_name", "estimate_name", "estimate_value")
    .unique()
    .sort("variable_name")
)

Combined estimate names:
shape: (13, 3)
┌─────────────────┬───────────────┬────────────────┐
│ variable_name   ┆ estimate_name ┆ estimate_value │
│ ---             ┆ ---           ┆ ---            │
│ str             ┆ str           ┆ str            │
╞═════════════════╪═══════════════╪════════════════╡
│ Medications     ┆ N (%)         ┆ 135 (35.8%)    │
│ Medications     ┆ N (%)         ┆ 153 (95.0%)    │
│ Medications     ┆ N (%)         ┆ 117 (77.5%)    │
│ Medications     ┆ N (%)         ┆ 149 (90.8%)    │
│ Medications     ┆ N (%)         ┆ 21 (22.3%)     │
│ …               ┆ …             ┆ …              │
│ age             ┆ Mean (SD)     ┆ 39.31 (14.03)  │
│ age             ┆ Mean (SD)     ┆ 34.45 (16.12)  │
│ age             ┆ Mean (SD)     ┆ 58.64 (15.52)  │
│ age             ┆ Mean (SD)     ┆ 47.97 (9.17)   │
│ number subjects ┆ count         ┆ 377            │
└─────────────────┴───────────────┴────────────────┘


### format_header

Pivots columns into multi-level column headers (for use with `great_tables` spanners).

```python
format_header(result, header, *, delim="\n", include_header_name=True, include_header_key=True)
```

In [8]:
# Prepare a tidy DataFrame first
tidy_df = tidy_result(result)
print("Tidy columns:", tidy_df.columns)
print(tidy_df.head(3))

Tidy columns: ['result_id', 'cdm_name', 'variable_name', 'variable_level', 'estimate_name', 'estimate_type', 'estimate_value', 'result_type', 'package_name', 'package_version', 'min_cell_count', 'cohort_name', 'age_group', 'sex']
shape: (3, 14)
┌───────────┬──────────┬─────────────┬────────────┬───┬────────────┬────────────┬───────────┬──────┐
│ result_id ┆ cdm_name ┆ variable_na ┆ variable_l ┆ … ┆ min_cell_c ┆ cohort_nam ┆ age_group ┆ sex  │
│ ---       ┆ ---      ┆ me          ┆ evel       ┆   ┆ ount       ┆ e          ┆ ---       ┆ ---  │
│ i64       ┆ str      ┆ ---         ┆ ---        ┆   ┆ ---        ┆ ---        ┆ str       ┆ str  │
│           ┆          ┆ str         ┆ str        ┆   ┆ str        ┆ str        ┆           ┆      │
╞═══════════╪══════════╪═════════════╪════════════╪═══╪════════════╪════════════╪═══════════╪══════╡
│ 1         ┆ mock_cdm ┆ number      ┆ number     ┆ … ┆ 5          ┆ cohort_1   ┆ null      ┆ null │
│           ┆          ┆ subjects    ┆ subjects 

In [9]:
# Pivot cohort_name into column headers
if "cohort_name" in tidy_df.columns:
    pivoted = format_header(tidy_df, ["cohort_name"])
    print("Pivoted columns:")
    for col in pivoted.columns:
        print(f"  {col!r}")
    print(pivoted.head(3))

Pivoted columns:
  'result_id'
  'cdm_name'
  'variable_name'
  'variable_level'
  'estimate_name'
  'estimate_type'
  '[header_name]cohort_name\n[header_level]cohort_1'
  'result_type'
  'package_name'
  'package_version'
  'min_cell_count'
  'age_group'
  'sex'
  '[header_name]cohort_name\n[header_level]cohort_2'
shape: (3, 14)
┌───────────┬──────────┬─────────────┬────────────┬───┬────────────┬───────────┬──────┬────────────┐
│ result_id ┆ cdm_name ┆ variable_na ┆ variable_l ┆ … ┆ min_cell_c ┆ age_group ┆ sex  ┆ [header_na │
│ ---       ┆ ---      ┆ me          ┆ evel       ┆   ┆ ount       ┆ ---       ┆ ---  ┆ me]cohort_ │
│ i64       ┆ str      ┆ ---         ┆ ---        ┆   ┆ ---        ┆ str       ┆ str  ┆ name       │
│           ┆          ┆ str         ┆ str        ┆   ┆ str        ┆           ┆      ┆ [head…     │
│           ┆          ┆             ┆            ┆   ┆            ┆           ┆      ┆ ---        │
│           ┆          ┆             ┆            ┆   ┆       

### customise_text

Transforms text strings for display — default converts snake_case to Sentence case.

```python
customise_text(x, *, fun=None, custom=None, keep=None)
```

In [10]:
# Default transform: snake_case -> Sentence case
print(customise_text("cohort_name"))
print(customise_text("variable_level"))
print(customise_text(["age_group", "sex", "cdm_name"]))

Cohort name
Variable level
['Age group', 'Sex', 'Cdm name']


In [11]:
# Custom replacements and keep list
print(customise_text("cdm_name", custom={"cdm_name": "Database"}))
print(customise_text("OMOP", keep=["OMOP"]))

Database
OMOP


## 4. Min Cell Count Suppression

`format_min_cell_count` reads the `min_cell_count` from the result's settings and
replaces suppressed values (`"-"`) with `"<N"` display strings.

```python
format_min_cell_count(result)  # no min_cell_count parameter — reads from settings
```

In [12]:
# First suppress small counts (min_cell_count=100 to force some suppression)
suppressed = result.suppress(min_cell_count=100)

# Check how many values are now "-"
n_suppressed = suppressed.data.filter(pl.col("estimate_value") == "-").shape[0]
print(f"Suppressed rows (sentinel '-'): {n_suppressed}")

print("\nBefore format_min_cell_count:")
print(
    suppressed.data.filter(pl.col("estimate_value") == "-")
    .select("variable_name", "estimate_name", "estimate_value")
    .head(5)
)

Suppressed rows (sentinel '-'): 1

Before format_min_cell_count:
shape: (1, 3)
┌─────────────────┬───────────────┬────────────────┐
│ variable_name   ┆ estimate_name ┆ estimate_value │
│ ---             ┆ ---           ┆ ---            │
│ str             ┆ str           ┆ str            │
╞═════════════════╪═══════════════╪════════════════╡
│ number subjects ┆ count         ┆ -              │
└─────────────────┴───────────────┴────────────────┘


In [13]:
# Apply format_min_cell_count — replaces "-" with "<5" (from settings)
display_suppressed = format_min_cell_count(suppressed)

print("After format_min_cell_count:")
print(
    display_suppressed.data.filter(pl.col("estimate_value").str.starts_with("<"))
    .select("variable_name", "estimate_name", "estimate_value")
    .head(5)
)

After format_min_cell_count:
shape: (1, 3)
┌─────────────────┬───────────────┬────────────────┐
│ variable_name   ┆ estimate_name ┆ estimate_value │
│ ---             ┆ ---           ┆ ---            │
│ str             ┆ str           ┆ str            │
╞═════════════════╪═══════════════╪════════════════╡
│ number subjects ┆ count         ┆ <5             │
└─────────────────┴───────────────┴────────────────┘


## 5. Table Styling

`TableStyle` controls the appearance of rendered tables. All fields have defaults.

In [14]:
# Default table style
ts = default_table_style()
print("Default TableStyle:")
print(f"  title_align:        {ts.title_align}")
print(f"  title_color:        {ts.title_color}")
print(f"  header_background:  {ts.header_background}")
print(f"  header_color:       {ts.header_color}")
print(f"  header_align:       {ts.header_align}")
print(f"  body_align:         {ts.body_align}")
print(f"  group_background:   {ts.group_background}")
print(f"  group_color:        {ts.group_color}")
print(f"  stripe:             {ts.stripe}")
print(f"  stripe_color:       {ts.stripe_color}")
print(f"  na_display:         {ts.na_display!r}")
print(f"  font_family:        {ts.font_family}")
print(f"  font_size:          {ts.font_size}")

Default TableStyle:
  title_align:        left
  title_color:        #333333
  header_background:  #4361ee
  header_color:       #ffffff
  header_align:       center
  body_align:         right
  group_background:   #e8eaf6
  group_color:        #333333
  stripe:             True
  stripe_color:       #f5f5f5
  na_display:         '–'
  font_family:        system-ui, -apple-system, sans-serif
  font_size:          14


In [15]:
# Custom table style
custom_ts = TableStyle(
    header_background="#2E86AB",
    header_color="#FFFFFF",
    stripe_color="#F0F8FF",
    font_size=13,
    na_display="—",
)
print(f"Custom header_background: {custom_ts.header_background}")
print(f"Custom font_size: {custom_ts.font_size}")

Custom header_background: #2E86AB
Custom font_size: 13


## 6. vis_omop_table

The main high-level table function. Executes the full format pipeline:
1. Format estimate values
2. Show min cell count markers
3. Format estimate names
4. Split name-level pairs, add settings
5. Header pivoting
6. Render to `great_tables.GT` or polars DataFrame

```python
vis_omop_table(result, *, estimate_name=None, header=None, type=None, style=None, ...)
```

In [16]:
# Basic table with type="polars" (always available, no great_tables needed)
tbl = vis_omop_table(result, type="polars")
print(f"Output type: {type(tbl).__name__}")
print(f"Columns: {tbl.columns}")
print(tbl.head(5))

Output type: DataFrame
Columns: ['cdm_name', 'variable_name', 'variable_level', 'estimate_name', 'estimate_value', 'cohort_name', 'age_group', 'sex']
shape: (5, 8)
┌──────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬───────────┬─────┐
│ cdm_name ┆ variable_na ┆ variable_le ┆ estimate_na ┆ estimate_va ┆ cohort_name ┆ age_group ┆ sex │
│ ---      ┆ me          ┆ vel         ┆ me          ┆ lue         ┆ ---         ┆ ---       ┆ --- │
│ str      ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ str         ┆ str       ┆ str │
│          ┆ str         ┆ str         ┆ str         ┆ str         ┆             ┆           ┆     │
╞══════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═══════════╪═════╡
│ mock_cdm ┆ number      ┆ number      ┆ count       ┆ 377         ┆ cohort_1    ┆ –         ┆ –   │
│          ┆ subjects    ┆ subjects    ┆             ┆             ┆             ┆           ┆     │
│ mock_cdm ┆ age         ┆ a

In [17]:
# With estimate_name combining and column grouping
tbl_combined = vis_omop_table(
    result,
    estimate_name={"N (%)": "<count> (<percentage>%)", "Mean (SD)": "<mean> (<sd>)"},
    type="polars",
)
print("Combined estimates table:")
print(tbl_combined.head(8))

Combined estimates table:
shape: (8, 8)
┌──────────┬─────────────┬─────────────┬────────────┬────────────┬────────────┬───────────┬────────┐
│ cdm_name ┆ variable_na ┆ variable_le ┆ estimate_n ┆ estimate_v ┆ cohort_nam ┆ age_group ┆ sex    │
│ ---      ┆ me          ┆ vel         ┆ ame        ┆ alue       ┆ e          ┆ ---       ┆ ---    │
│ str      ┆ ---         ┆ ---         ┆ ---        ┆ ---        ┆ ---        ┆ str       ┆ str    │
│          ┆ str         ┆ str         ┆ str        ┆ str        ┆ str        ┆           ┆        │
╞══════════╪═════════════╪═════════════╪════════════╪════════════╪════════════╪═══════════╪════════╡
│ mock_cdm ┆ Medications ┆ Amoxiciline ┆ N (%)      ┆ 135        ┆ cohort_1   ┆ –         ┆ –      │
│          ┆             ┆             ┆            ┆ (35.8%)    ┆            ┆           ┆        │
│ mock_cdm ┆ Medications ┆ Amoxiciline ┆ N (%)      ┆ 149        ┆ cohort_1   ┆ <40       ┆ Male   │
│          ┆             ┆             ┆           

In [18]:
# With custom style and title
tbl_styled = vis_omop_table(
    result,
    estimate_name={"N (%)": "<count> (<percentage>%)"},
    style=custom_ts,
    title="Mock Cohort Summary",
    subtitle="Generated from mock_summarised_result",
    type="polars",
)
print(tbl_styled.head(5))

shape: (5, 8)
┌──────────┬─────────────┬─────────────┬────────────┬────────────┬────────────┬───────────┬────────┐
│ cdm_name ┆ variable_na ┆ variable_le ┆ estimate_n ┆ estimate_v ┆ cohort_nam ┆ age_group ┆ sex    │
│ ---      ┆ me          ┆ vel         ┆ ame        ┆ alue       ┆ e          ┆ ---       ┆ ---    │
│ str      ┆ ---         ┆ ---         ┆ ---        ┆ ---        ┆ ---        ┆ str       ┆ str    │
│          ┆ str         ┆ str         ┆ str        ┆ str        ┆ str        ┆           ┆        │
╞══════════╪═════════════╪═════════════╪════════════╪════════════╪════════════╪═══════════╪════════╡
│ mock_cdm ┆ Medications ┆ Amoxiciline ┆ N (%)      ┆ 135        ┆ cohort_1   ┆ —         ┆ —      │
│          ┆             ┆             ┆            ┆ (35.8%)    ┆            ┆           ┆        │
│ mock_cdm ┆ Medications ┆ Amoxiciline ┆ N (%)      ┆ 149        ┆ cohort_1   ┆ <40       ┆ Male   │
│          ┆             ┆             ┆            ┆ (90.8%)    ┆           

In [19]:
# With header pivoting — cohort_name becomes column headers
tbl_header = vis_omop_table(
    result,
    estimate_name={"N (%)": "<count> (<percentage>%)"},
    header=["cohort_name"],
    type="polars",
)
print("With header pivoting on cohort_name:")
print(f"Columns: {tbl_header.columns}")
print(tbl_header.head(5))

With header pivoting on cohort_name:
Columns: ['cdm_name', 'variable_name', 'variable_level', 'estimate_name', '[header_name]cohort_name\n[header_level]cohort_1', 'age_group', 'sex', '[header_name]cohort_name\n[header_level]cohort_2']
shape: (5, 8)
┌──────────┬─────────────┬─────────────┬────────────┬────────────┬───────────┬────────┬────────────┐
│ cdm_name ┆ variable_na ┆ variable_le ┆ estimate_n ┆ [header_na ┆ age_group ┆ sex    ┆ [header_na │
│ ---      ┆ me          ┆ vel         ┆ ame        ┆ me]cohort_ ┆ ---       ┆ ---    ┆ me]cohort_ │
│ str      ┆ ---         ┆ ---         ┆ ---        ┆ name       ┆ str       ┆ str    ┆ name       │
│          ┆ str         ┆ str         ┆ str        ┆ [head…     ┆           ┆        ┆ [head…     │
│          ┆             ┆             ┆            ┆ ---        ┆           ┆        ┆ ---        │
│          ┆             ┆             ┆            ┆ str        ┆           ┆        ┆ str        │
╞══════════╪═════════════╪═════════════╪════

In [20]:
# Using mock_cohort_characteristics for a richer table
tbl_chars = vis_omop_table(
    chars,
    estimate_name={
        "N (%)": "<count> (<percentage>%)",
        "Mean (SD)": "<mean> (<sd>)",
    },
    type="polars",
    title="Cohort Characteristics",
)
print("Characteristics table:")
print(tbl_chars.head(10))

Characteristics table:
shape: (10, 7)
┌──────────┬───────────────┬────────────────┬───────────────┬───────────────┬─────────────┬────────┐
│ cdm_name ┆ variable_name ┆ variable_level ┆ estimate_name ┆ estimate_valu ┆ cohort_name ┆ sex    │
│ ---      ┆ ---           ┆ ---            ┆ ---           ┆ e             ┆ ---         ┆ ---    │
│ str      ┆ str           ┆ str            ┆ str           ┆ ---           ┆ str         ┆ str    │
│          ┆               ┆                ┆               ┆ str           ┆             ┆        │
╞══════════╪═══════════════╪════════════════╪═══════════════╪═══════════════╪═════════════╪════════╡
│ mock_cdm ┆ Sex           ┆ Male           ┆ N (%)         ┆ 170 (45.1%)   ┆ cohort_1    ┆ –      │
│ mock_cdm ┆ Sex           ┆ Female         ┆ N (%)         ┆ 207 (54.9%)   ┆ cohort_1    ┆ –      │
│ mock_cdm ┆ Sex           ┆ Male           ┆ N (%)         ┆ 135 (31.5%)   ┆ cohort_1    ┆ Male   │
│ mock_cdm ┆ Sex           ┆ Female         ┆ N (%)  

## 7. vis_table

A lower-level function that works with any Polars DataFrame (not just `SummarisedResult`).

```python
vis_table(result, *, header=None, group_column=None, type=None, style=None, ...)
```

In [21]:
# Create a simple polars DataFrame
df = pl.DataFrame({
    "cohort": ["A", "A", "B", "B"],
    "variable": ["age", "weight", "age", "weight"],
    "mean": [45.2, 72.1, 52.8, 68.5],
    "sd": [12.3, 15.6, 10.1, 14.2],
})

tbl = vis_table(df, type="polars", title="Simple DataFrame Table")
print(tbl)

shape: (4, 4)
┌────────┬──────────┬──────┬──────┐
│ cohort ┆ variable ┆ mean ┆ sd   │
│ ---    ┆ ---      ┆ ---  ┆ ---  │
│ str    ┆ str      ┆ str  ┆ str  │
╞════════╪══════════╪══════╪══════╡
│ A      ┆ age      ┆ 45.2 ┆ 12.3 │
│ A      ┆ weight   ┆ 72.1 ┆ 15.6 │
│ B      ┆ age      ┆ 52.8 ┆ 10.1 │
│ B      ┆ weight   ┆ 68.5 ┆ 14.2 │
└────────┴──────────┴──────┴──────┘


In [22]:
# With column hiding and renaming
tbl_renamed = vis_table(
    df,
    rename={"Cohort Name": "cohort", "Variable": "variable"},
    hide=["sd"],
    type="polars",
)
print("Renamed and hidden columns:")
print(tbl_renamed)

Renamed and hidden columns:
shape: (4, 3)
┌─────────────┬──────────┬──────┐
│ Cohort Name ┆ Variable ┆ mean │
│ ---         ┆ ---      ┆ ---  │
│ str         ┆ str      ┆ str  │
╞═════════════╪══════════╪══════╡
│ A           ┆ age      ┆ 45.2 │
│ A           ┆ weight   ┆ 72.1 │
│ B           ┆ age      ┆ 52.8 │
│ B           ┆ weight   ┆ 68.5 │
└─────────────┴──────────┴──────┘


## 8. format_table

Low-level rendering function. Works directly on a `DataFrame`.

```python
format_table(x, *, type=None, style=None, na=None, title=None, subtitle=None,
             group_column=None, group_as_column=False, merge="all_columns")
```

In [23]:
# format_table with group_column
grouped_df = pl.DataFrame({
    "group": ["Demographics", "Demographics", "Lab Values", "Lab Values"],
    "variable": ["Age", "Sex (% Female)", "HbA1c", "Creatinine"],
    "value": ["52.3 (14.1)", "55.2%", "6.8 (1.2)", "1.1 (0.3)"],
})

tbl = format_table(
    grouped_df,
    type="polars",
    group_column=["group"],
    title="Patient Summary",
)
print(tbl)

shape: (4, 3)
┌──────────────┬────────────────┬─────────────┐
│ group        ┆ variable       ┆ value       │
│ ---          ┆ ---            ┆ ---         │
│ str          ┆ str            ┆ str         │
╞══════════════╪════════════════╪═════════════╡
│ Demographics ┆ Age            ┆ 52.3 (14.1) │
│ Demographics ┆ Sex (% Female) ┆ 55.2%       │
│ Lab Values   ┆ HbA1c          ┆ 6.8 (1.2)   │
│ Lab Values   ┆ Creatinine     ┆ 1.1 (0.3)   │
└──────────────┴────────────────┴─────────────┘


In [24]:
# format_table with custom NA display
df_with_nulls = pl.DataFrame({
    "name": ["Alice", "Bob", "Carol"],
    "score": ["95", None, "88"],
    "grade": ["A", "B", None],
})

tbl = format_table(df_with_nulls, type="polars", na="N/A")
print("Custom NA display:")
print(tbl)

Custom NA display:
shape: (3, 3)
┌───────┬───────┬───────┐
│ name  ┆ score ┆ grade │
│ ---   ┆ ---   ┆ ---   │
│ str   ┆ str   ┆ str   │
╞═══════╪═══════╪═══════╡
│ Alice ┆ 95    ┆ A     │
│ Bob   ┆ N/A   ┆ B     │
│ Carol ┆ 88    ┆ N/A   │
└───────┴───────┴───────┘


## 9. Plot Styling

`PlotStyle` controls the appearance of plotly figures. All fields have defaults.

In [25]:
# Default plot style
ps = default_plot_style()
print("Default PlotStyle:")
print(f"  color_palette:    {ps.color_palette[:4]}... ({len(ps.color_palette)} colours)")
print(f"  background_color: {ps.background_color}")
print(f"  text_color:       {ps.text_color}")
print(f"  grid_color:       {ps.grid_color}")
print(f"  font_family:      {ps.font_family}")
print(f"  font_size:        {ps.font_size}")
print(f"  title_size:       {ps.title_size}")
print(f"  show_legend:      {ps.show_legend}")

Default PlotStyle:
  color_palette:    ['#4361ee', '#3a86ff', '#8338ec', '#ff006e']... (10 colours)
  background_color: #ffffff
  text_color:       #333333
  grid_color:       #e0e0e0
  font_family:      system-ui, -apple-system, sans-serif
  font_size:        12
  title_size:       16
  show_legend:      True


In [26]:
# Custom plot style
custom_ps = PlotStyle(
    color_palette=["#E63946", "#457B9D", "#2A9D8F", "#E9C46A"],
    background_color="#FAFAFA",
    text_color="#264653",
    grid_color="#E0E0E0",
    font_family="Helvetica",
    font_size=11,
    title_size=15,
    show_legend=True,
)
print(f"Custom palette: {custom_ps.color_palette}")
print(f"Custom background: {custom_ps.background_color}")

Custom palette: ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']
Custom background: #FAFAFA


## 10. Scatter Plot

```python
scatter_plot(result, *, x, y, line=False, point=True, ribbon=False,
             y_min=None, y_max=None, facet=None, colour=None, style=None,
             group=None, title=None, x_title=None, y_title=None)
```

Accepts a `SummarisedResult` (auto-tidied and pivoted) or a Polars DataFrame.

In [27]:
# Scatter plot from a plain DataFrame
scatter_df = pl.DataFrame({
    "age": [30, 35, 40, 45, 50, 55, 60, 65, 70, 75],
    "count": [120, 150, 200, 250, 310, 280, 220, 180, 140, 100],
    "cohort": ["A", "B", "A", "B", "A", "B", "A", "B", "A", "B"],
})

fig = scatter_plot(
    scatter_df,
    x="age",
    y="count",
    colour="cohort",
    title="Subject Count by Age",
    x_title="Age",
    y_title="Count",
)
print(f"Figure type: {type(fig).__name__}")
print(f"Number of traces: {len(fig.data)}")
fig.show()

Figure type: Figure
Number of traces: 2


In [28]:
# Scatter plot with lines and custom style
fig_line = scatter_plot(
    scatter_df,
    x="age",
    y="count",
    colour="cohort",
    line=True,
    point=True,
    style=custom_ps,
    title="Count Trend by Age",
)
fig_line.show()

In [29]:
# Scatter with ribbon (confidence interval)
ribbon_df = pl.DataFrame({
    "year": [2018, 2019, 2020, 2021, 2022, 2018, 2019, 2020, 2021, 2022],
    "prevalence": [5.2, 5.5, 6.1, 6.8, 7.2, 3.1, 3.4, 3.9, 4.2, 4.5],
    "lower": [4.8, 5.1, 5.6, 6.3, 6.7, 2.7, 3.0, 3.4, 3.8, 4.1],
    "upper": [5.6, 5.9, 6.6, 7.3, 7.7, 3.5, 3.8, 4.4, 4.6, 4.9],
    "condition": ["Diabetes"] * 5 + ["Hypertension"] * 5,
})

fig_ribbon = scatter_plot(
    ribbon_df,
    x="year",
    y="prevalence",
    y_min="lower",
    y_max="upper",
    colour="condition",
    line=True,
    ribbon=True,
    title="Prevalence Over Time (with CI)",
)
fig_ribbon.show()

## 11. Bar Plot

```python
bar_plot(result, *, x, y, position="dodge", facet=None, colour=None,
         style=None, title=None, x_title=None, y_title=None)
```

In [30]:
# Bar plot from a DataFrame
bar_df = pl.DataFrame({
    "cohort": ["Cohort A", "Cohort A", "Cohort B", "Cohort B"],
    "variable": ["Male", "Female", "Male", "Female"],
    "count": [145, 167, 198, 212],
})

fig = bar_plot(
    bar_df,
    x="variable",
    y="count",
    colour="cohort",
    title="Subject Counts by Sex",
)
print(f"Figure type: {type(fig).__name__}")
fig.show()

Figure type: Figure


In [31]:
# Stacked bar plot with faceting
facet_df = pl.DataFrame({
    "age_group": ["<40", "<40", "<40", "<40", ">=40", ">=40", ">=40", ">=40"],
    "sex": ["Male", "Female", "Male", "Female", "Male", "Female", "Male", "Female"],
    "cohort": ["A", "A", "B", "B", "A", "A", "B", "B"],
    "count": [50, 60, 45, 55, 80, 90, 75, 85],
})

fig_stacked = bar_plot(
    facet_df,
    x="sex",
    y="count",
    colour="cohort",
    facet="age_group",
    position="stack",
    style=custom_ps,
    title="Counts by Sex, Faceted by Age Group",
)
fig_stacked.show()

## 12. Box Plot

Renders from pre-computed summary statistics (not raw data).

```python
box_plot(result, *, x, lower="q25", middle="median", upper="q75",
         y_min="min", y_max="max", facet=None, colour=None, style=None, ...)
```

Takes `lower`, `middle`, `upper`, `y_min`, `y_max` — NOT `x, y`.

In [32]:
# Box plot from pre-computed quantiles
box_df = pl.DataFrame({
    "cohort": ["Cohort A", "Cohort B", "Cohort C"],
    "min": [18.0, 22.0, 15.0],
    "q25": [35.0, 38.0, 30.0],
    "median": [48.0, 52.0, 45.0],
    "q75": [62.0, 65.0, 58.0],
    "max": [85.0, 88.0, 82.0],
})

fig = box_plot(
    box_df,
    x="cohort",
    lower="q25",
    middle="median",
    upper="q75",
    y_min="min",
    y_max="max",
    title="Age Distribution by Cohort",
    y_title="Age (years)",
)
print(f"Traces: {len(fig.data)}")
fig.show()

Traces: 1


In [33]:
# Box plot with colour grouping
box_colour_df = pl.DataFrame({
    "variable": ["Age", "Age", "Age", "Age", "Prior Obs", "Prior Obs", "Prior Obs", "Prior Obs"],
    "sex": ["Male", "Female", "Male", "Female", "Male", "Female", "Male", "Female"],
    "min": [20.0, 18.0, 100.0, 120.0, 20.0, 18.0, 100.0, 120.0],
    "q25": [35.0, 33.0, 500.0, 550.0, 35.0, 33.0, 500.0, 550.0],
    "median": [50.0, 48.0, 1200.0, 1300.0, 50.0, 48.0, 1200.0, 1300.0],
    "q75": [63.0, 60.0, 2000.0, 2100.0, 63.0, 60.0, 2000.0, 2100.0],
    "max": [85.0, 82.0, 3500.0, 3600.0, 85.0, 82.0, 3500.0, 3600.0],
})

fig = box_plot(
    box_colour_df,
    x="variable",
    lower="q25",
    middle="median",
    upper="q75",
    y_min="min",
    y_max="max",
    colour="sex",
    style=custom_ps,
    title="Summary Statistics by Variable and Sex",
)
fig.show()

## 13. Customising Text

`customise_text` is used internally by plot/table functions to format labels.
You can also use it directly with custom transformations.

In [34]:
# Default: snake_case -> Sentence case
labels = ["variable_name", "estimate_value", "cohort_name", "age_group"]
formatted_labels = customise_text(labels)
for orig, fmt in zip(labels, formatted_labels):
    print(f"  {orig:20s} -> {fmt}")

  variable_name        -> Variable name
  estimate_value       -> Estimate value
  cohort_name          -> Cohort name
  age_group            -> Age group


In [35]:
# Custom function
result_upper = customise_text(
    ["cohort_name", "age_group"],
    fun=lambda s: s.replace("_", " ").upper(),
)
print("Custom uppercase transform:", result_upper)

Custom uppercase transform: ['COHORT NAME', 'AGE GROUP']


In [36]:
# Mix of custom replacements and keep
result_mixed = customise_text(
    ["cdm_name", "OMOP", "variable_level"],
    custom={"cdm_name": "Database"},
    keep=["OMOP"],
)
print("Mixed transform:", result_mixed)

Mixed transform: ['Database', 'OMOP', 'Variable level']


## 14. Combining Styles — Full Example

Putting it all together: create a custom-styled table and plot from the same data.

In [37]:
# Custom styles for a publication-ready look
pub_table_style = TableStyle(
    header_background="#1B4332",
    header_color="#FFFFFF",
    group_background="#D8F3DC",
    group_color="#1B4332",
    stripe=True,
    stripe_color="#F0FFF0",
    font_size=12,
    na_display="—",
)

pub_plot_style = PlotStyle(
    color_palette=["#1B4332", "#2D6A4F", "#52B788", "#95D5B2"],
    background_color="#FFFFFF",
    text_color="#1B4332",
    grid_color="#E8E8E8",
    font_family="Georgia",
    font_size=12,
    title_size=16,
)

print("Publication styles created.")
print(f"Table header: {pub_table_style.header_background}")
print(f"Plot palette: {pub_plot_style.color_palette}")

Publication styles created.
Table header: #1B4332
Plot palette: ['#1B4332', '#2D6A4F', '#52B788', '#95D5B2']


In [38]:
# Use characteristics mock for richer data
chars_rich = mock_cohort_characteristics(n_cohorts=3, n_strata=1)

# Styled table
pub_table = vis_omop_table(
    chars_rich,
    estimate_name={
        "N (%)": "<count> (<percentage>%)",
        "Mean (SD)": "<mean> (<sd>)",
    },
    style=pub_table_style,
    title="Cohort Characteristics — Publication Style",
    type="polars",
)
print(pub_table.head(10))

shape: (10, 7)
┌──────────┬───────────────┬────────────────┬───────────────┬───────────────┬─────────────┬────────┐
│ cdm_name ┆ variable_name ┆ variable_level ┆ estimate_name ┆ estimate_valu ┆ cohort_name ┆ sex    │
│ ---      ┆ ---           ┆ ---            ┆ ---           ┆ e             ┆ ---         ┆ ---    │
│ str      ┆ str           ┆ str            ┆ str           ┆ ---           ┆ str         ┆ str    │
│          ┆               ┆                ┆               ┆ str           ┆             ┆        │
╞══════════╪═══════════════╪════════════════╪═══════════════╪═══════════════╪═════════════╪════════╡
│ mock_cdm ┆ Sex           ┆ Male           ┆ N (%)         ┆ 170 (45.1%)   ┆ cohort_1    ┆ —      │
│ mock_cdm ┆ Sex           ┆ Female         ┆ N (%)         ┆ 207 (54.9%)   ┆ cohort_1    ┆ —      │
│ mock_cdm ┆ Sex           ┆ Male           ┆ N (%)         ┆ 135 (31.5%)   ┆ cohort_1    ┆ Male   │
│ mock_cdm ┆ Sex           ┆ Female         ┆ N (%)         ┆ 294 (68.5%)   

In [39]:
# Styled bar plot from the same data — extract counts by cohort
count_data = chars_rich.data.filter(
    (pl.col("variable_name") == "Number subjects")
    & (pl.col("strata_name") == "overall")
).select(
    pl.col("group_level").alias("cohort"),
    pl.col("estimate_value").alias("count"),
)

fig = bar_plot(
    count_data,
    x="cohort",
    y="count",
    style=pub_plot_style,
    title="Number of Subjects per Cohort",
    x_title="Cohort",
    y_title="N Subjects",
)
fig.show()

## 15. Tidy Helpers

`tidy_result` and `tidy_columns` convert a `SummarisedResult` into a wide DataFrame
by splitting name-level pairs and adding settings columns.

In [40]:
# tidy_columns shows what columns the tidy form will have
cols = tidy_columns(result)
print(f"Tidy columns ({len(cols)}):")
for c in cols:
    print(f"  {c}")

Tidy columns (14):
  result_id
  cdm_name
  variable_name
  variable_level
  estimate_name
  estimate_type
  estimate_value
  result_type
  package_name
  package_version
  min_cell_count
  cohort_name
  age_group
  sex


In [41]:
# tidy_result converts to a wide tidy DataFrame
tidy = tidy_result(result)
print(f"Tidy shape: {tidy.shape}")
print(tidy.head(5))

Tidy shape: (30, 14)
shape: (5, 14)
┌───────────┬──────────┬─────────────┬────────────┬───┬────────────┬────────────┬───────────┬──────┐
│ result_id ┆ cdm_name ┆ variable_na ┆ variable_l ┆ … ┆ min_cell_c ┆ cohort_nam ┆ age_group ┆ sex  │
│ ---       ┆ ---      ┆ me          ┆ evel       ┆   ┆ ount       ┆ e          ┆ ---       ┆ ---  │
│ i64       ┆ str      ┆ ---         ┆ ---        ┆   ┆ ---        ┆ ---        ┆ str       ┆ str  │
│           ┆          ┆ str         ┆ str        ┆   ┆ str        ┆ str        ┆           ┆      │
╞═══════════╪══════════╪═════════════╪════════════╪═══╪════════════╪════════════╪═══════════╪══════╡
│ 1         ┆ mock_cdm ┆ number      ┆ number     ┆ … ┆ 5          ┆ cohort_1   ┆ null      ┆ null │
│           ┆          ┆ subjects    ┆ subjects   ┆   ┆            ┆            ┆           ┆      │
│ 1         ┆ mock_cdm ┆ age         ┆ age        ┆ … ┆ 5          ┆ cohort_1   ┆ null      ┆ null │
│ 1         ┆ mock_cdm ┆ age         ┆ age        ┆ … ┆

## 16. Summary

The `omopy.vis` module exports 19 objects:

| Category | Exports |
|---|---|
| **Format pipeline** | `format_estimate_value`, `format_estimate_name`, `format_header`, `format_min_cell_count`, `format_table` |
| **Tidy helpers** | `tidy_columns`, `tidy_result` |
| **High-level tables** | `vis_omop_table`, `vis_table` |
| **Plots** | `scatter_plot`, `bar_plot`, `box_plot` |
| **Style** | `TableStyle`, `PlotStyle`, `customise_text`, `default_table_style`, `default_plot_style` |
| **Mock data** | `mock_summarised_result` |

Key points:
- **No database needed** — `mock_summarised_result()` and `mock_cohort_characteristics()` generate test data
- **Format pipeline is composable** — chain `format_estimate_value` -> `format_estimate_name` -> `format_min_cell_count`
- **`vis_omop_table`** is the main entry point for styled tables (runs the full pipeline)
- **Plot functions** accept `SummarisedResult` (auto-tidied) or plain `DataFrame`
- **`box_plot`** takes `lower`/`middle`/`upper`/`y_min`/`y_max` — not raw data
- **Styles are frozen dataclasses** — create new instances for customisation
- **`customise_text`** converts snake_case to Sentence case by default, with custom/keep overrides